# Validation of the baseline panel with known asset-pricing results

If fundamentals, units and timing of the baseline panel are right, well-known results must appear. The
functions live in `src/datastream/validation.py`; this notebook configures and displays. Outputs go to
`OUTPUT_DIR`.

| Section | Test | What to expect |
|---|---|---|
| 1 | Fama-French factors rebuilt from the panel (2x3, NYSE breakpoints, June) vs. Ken French's factors | corr SMB >= 0.95, HML ~0.85-0.95, RMW/CMA ~0.8-0.9 |
| 2 | Fama-MacBeth regressions of next-month returns on characteristics | signs as in the literature (column `expected`) |
| 3 | Decile sorts, EW and VW | spreads with the same sign, roughly monotonic |
| 4 | Event study around report months, by earnings growth | most of the reaction before the availability month (report + lag); small drift after |

Column names are the readable names of the baseline panel (`total_assets`, `common_equity`, `market_cap`, ...;
mapping in `config/firm_variables.csv` and the JSON sidecar of the panel).

**Prerequisites** (Windows machine):
```
uv run python scripts/05_build_monthly_universe.py
uv run python scripts/06_build_baseline_panel.py                     # baseline_rolling_0.2
uv run python scripts/06_build_baseline_panel.py --convention ff     # baseline_ff_0.2 (section 1)
```
Literature benchmarks are approximate (other databases, samples and definitions); look for agreement in sign
and rough magnitude, and for large deviations.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from datastream import firm_evaluation as fe
from datastream import validation as va
from datastream.ff_data import get_ff_factors
from datastream.preprocessing.firm_data import read_registry

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
fe.set_style()

# ── configuration ───────────────────────────────────────────────────────────────────────────────────
REGION = "US"
FIRM_ROOT = Path(os.environ.get("DS_FIRM_ROOT", "D:/Datastream/Firmcharacteristics_Monthly")) / REGION
PRICE_PATH = Path(os.environ.get("DS_PRICE_PATH", "D:/Datastream/PriceData/US/processed"))
PENNY_PERCENTILE = "0.2"
BASELINE_ROLLING = FIRM_ROOT / "Paneldata" / "baseline" / f"baseline_rolling_{PENNY_PERCENTILE}.parquet"
BASELINE_FF = FIRM_ROOT / "Paneldata" / "baseline" / f"baseline_ff_{PENNY_PERCENTILE}.parquet"
STATICS_FILE = PRICE_PATH / f"statics_filtered_{PENNY_PERCENTILE}.csv"     # EXMNEM for NYSE breakpoints
INTEREST_VAR = None          # "interest_expense" (WC01251) once downloaded; None: OP = operating income / BE
LAG_MONTHS = 3               # the lag used for BASELINE_ROLLING (marked in the event study)
HAC_LAGS = 6
SUBPERIODS = [("1994-2004", "1994-01", "2004-12"), ("2005-2014", "2005-01", "2014-12"), ("2015-", "2015-01", "2099-12")]
OUTPUT_DIR = FIRM_ROOT / "Paneldata" / "validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
registry = read_registry(REPO / "config" / "firm_variables.csv")
signs = dict(zip(registry["variable"], registry.get("sign", pd.Series("", index=registry.index))))

C_MINE, C_FF = fe.CATEGORICAL[1], fe.CATEGORICAL[0]

## 0. Load

In [ ]:
panel = pd.read_parquet(BASELINE_ROLLING)
print(f"Rolling panel: {len(panel):,} stock-months, {panel['DSCD'].nunique():,} stocks, "
      f"{panel['Date'].min():%Y-%m} to {panel['Date'].max():%Y-%m}")
statics = pd.read_csv(STATICS_FILE, dtype=str)
nyse = set(statics.loc[statics["EXMNEM"].str.upper().str.startswith("NYS", na=False), "DSCD"].str.strip())
print(f"NYSE stocks (current exchange from the statics): {len(nyse):,}")

ff3 = get_ff_factors(3, "monthly")
ff5 = get_ff_factors(5, "monthly")

## 1. Fama-French factors rebuilt from the panel
Built from the panel with the Fama-French timing (`--convention ff`): portfolios formed at the end of June t
with market cap of June t; B/M = book equity of fiscal year t-1 / market cap of December t-1; OP = operating
income (minus interest expense if `INTEREST_VAR` is set) / book equity; INV = asset growth; NYSE breakpoints
(median size; 30th/70th percentiles); value-weighted returns July t to June t+1.

Known differences to Ken French's construction: Worldscope common equity instead of Compustat book equity
(deferred taxes, preferred stock), Datastream universe and filters instead of CRSP, current instead of
historical exchange listing for the NYSE breakpoints, and the OP approximation.

In [ ]:
if BASELINE_FF.exists():
    panel_ff = pd.read_parquet(BASELINE_FF)
    factors = va.replicate_ff_factors(panel_ff, nyse, INTEREST_VAR)
    cmp3 = va.compare_factors(factors, ff3, {"SMB_BM": "SMB", "HML": "HML"}, HAC_LAGS)
    cmp5 = va.compare_factors(factors, ff5, {"SMB": "SMB", "RMW": "RMW", "CMA": "CMA"}, HAC_LAGS)
    comparison = pd.concat([cmp3.assign(model="FF3"), cmp5.assign(model="FF5")])
    comparison.to_csv(OUTPUT_DIR / "1_ff_factor_comparison.csv")
    factors.to_csv(OUTPUT_DIR / "1_replicated_factors.csv")
    display(comparison[["model", "french", "months", "corr", "mean_mine_annual", "mean_ff_annual", "t_mine", "t_ff",
                        "vol_mine_annual", "vol_ff_annual", "slope", "tracking_error_annual"]])
else:
    factors = None
    print(f"{BASELINE_FF.name} not found: run scripts/06_build_baseline_panel.py --convention ff")

In [ ]:
if factors is not None:
    pairs = [("SMB_BM", ff3, "SMB"), ("HML", ff3, "HML"), ("RMW", ff5, "RMW"), ("CMA", ff5, "CMA")]
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=False)
    for ax, (mine, ref, col) in zip(axes, pairs):
        x = pd.concat([factors[mine].rename("rebuilt"), ref[col].rename("Ken French")], axis=1, join="inner").dropna()
        cum = np.log1p(x).cumsum()
        idx = cum.index.to_timestamp()
        ax.plot(idx, cum["Ken French"], color=C_FF, label="Ken French")
        ax.plot(idx, cum["rebuilt"], color=C_MINE, label="rebuilt from panel")
        ax.set_title(f"{col}  (corr {x['rebuilt'].corr(x['Ken French']):.2f})", loc="left")
    axes[0].set_ylabel("cumulative log return")
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.06))
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "1_factors_cumulative.png", dpi=130, bbox_inches="tight")
    plt.show()

## 2. Fama-MacBeth regressions
Monthly cross-sectional regressions of the return in month t+1 on characteristics known at the end of month t
(from the rolling panel: fundamentals available after the lag). Characteristics are winsorized at 1%/99% and
standardized per month, so coefficients are monthly returns per one cross-sectional standard deviation.
Newey-West t-statistics. Samples: all stocks, and without the smallest NYSE size quintile.

In [ ]:
d = va.characteristics(panel, INTEREST_VAR)
xs = [c for c in va.CHARACTERISTICS if c in d.columns and d[c].notna().any()]
print("Characteristics:", xs)
print(d[xs].describe(percentiles=[0.01, 0.5, 0.99]).T[["count", "mean", "1%", "50%", "99%"]])

fm = {}
fm["all"], fm_coefs = va.fama_macbeth(d, xs, hac_lags=HAC_LAGS)
fm["no smallest quintile"], _ = va.fama_macbeth(d[d["size_group"] > 1], xs, hac_lags=HAC_LAGS)
for label, a, b in SUBPERIODS:
    fm[label], _ = va.fama_macbeth(d, xs, hac_lags=HAC_LAGS, start=a, end=pd.Period(b).end_time)
fm_table = pd.concat({k: v[["coef", "t_nw"]] for k, v in fm.items()}, axis=1)
fm_table.insert(0, "expected", fm["all"]["expected"])
fm_table.to_csv(OUTPUT_DIR / "2_fama_macbeth.csv")
print(f"All stocks: {fm['all'].attrs['avg_n']:.0f} stocks per month, average R2 {fm['all'].attrs['avg_r2']:.3f}")
display(fm_table)

## 3. Decile sorts
Monthly deciles on each characteristic (all-stock breakpoints), equal- and value-weighted returns of month
t+1, and the D10-D1 spread with Newey-West t. A spread that only shows up equal-weighted is a small-cap effect.

In [ ]:
sort_rows = {}
for c in xs:
    tab, _ = va.decile_sorts(d, c, hac_lags=HAC_LAGS)
    for w in ["EW", "VW"]:
        sort_rows[(c, w)] = tab.loc[w]
sorts = pd.DataFrame(sort_rows).T
sorts.index.names = ["characteristic", "weighting"]
sorts.to_csv(OUTPUT_DIR / "3_decile_sorts.csv")
display(sorts)

## 4. Event study around report months
For annual reports (previous report 9-15 months earlier): earnings growth = (NI - NI_prev) / TA_prev, quintiles
per report year. Market-adjusted returns (minus the equal-weighted universe mean) from 3 months before to 12
months after the report month (month 0 = when Datastream changes the value, about one month after fiscal year
end). The dashed line marks the availability month of the rolling panel (report month + `LAG_MONTHS`).

Expected: earnings announcements and 10-K filings fall in the first months after fiscal year end, so most of
the return difference between high and low earnings growth should appear in months 0 to `LAG_MONTHS`-1 - which
the rolling panel does not use - and only a small drift (post-earnings-announcement drift) afterwards. A large
spread after the availability month would suggest a longer lag.

In [ ]:
snaps = va.report_events(FIRM_ROOT, panel[["DSCD"]].drop_duplicates(), signs)
ev = va.event_study(snaps, panel[["DSCD", "Date", "ret"]], k_min=-3, k_max=12)
print(f"{ev['n_events']:,} annual report events")
cum = ev["by_group"].loc[-3:].cumsum()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
ramp = [fe.BLUE_RAMP[i] for i in (1, 2, 3, 5, 6)]
for g, colr in zip(cum.columns, ramp):
    axes[0].plot(cum.index, cum[g], color=colr, marker="o", ms=3, label=f"Q{int(g)}")
axes[0].set_title("Cumulative market-adjusted return by earnings-growth quintile", loc="left")
axes[0].legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.18), title="Q1 = lowest growth")
sp = ev["spread"]
axes[1].bar(sp.index, sp["spread"], color=fe.CATEGORICAL[0], width=0.7)
axes[1].set_title("Q5 - Q1 monthly spread by event month", loc="left")
for ax in axes:
    ax.axvline(LAG_MONTHS - 0.5, color=fe.INK_MUTED, ls="--", lw=1)
    ax.axhline(0, color=fe.GRID, lw=1)
    ax.set_xlabel("months relative to report month")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "4_event_study.png", dpi=130, bbox_inches="tight")
plt.show()

windows = {"before report (-3..-1)": (-3, -1), f"report to availability (0..{LAG_MONTHS - 1})": (0, LAG_MONTHS - 1),
           f"after availability ({LAG_MONTHS}..{LAG_MONTHS + 2})": (LAG_MONTHS, LAG_MONTHS + 2),
           f"later ({LAG_MONTHS + 3}..12)": (LAG_MONTHS + 3, 12)}
win = va.window_returns(sp, windows)
display(sp.T)
display(win)
sp.to_csv(OUTPUT_DIR / "4_event_spread.csv")
win.to_csv(OUTPUT_DIR / "4_event_windows.csv")